# 🔒 Senthium AI: Dynamic Face Recognition with AI Training\n
\n
**CSE 311 - Artificial Intelligence Project**\n
\n
**Student:** [Your Name]  \n
**Register Number:** [Your Number]  \n
**Date:** November 13, 2025\n
\n
---\n
\n
## 🎯 Project Overview\n
\n
**Senthium AI** demonstrates a complete AI pipeline for facial recognition security:\n
\n
### AI Techniques Implemented:\n
1. **Deep Convolutional Neural Networks** (FaceNet architecture)\n
2. **Transfer Learning** (Pre-trained on VGGFace2 - 3.31M images)\n
3. **Dynamic Training** (1-minute video capture)\n
4. **Temporal Face Learning** (Multi-angle, multi-lighting)\n
5. **Embedding Fine-Tuning** (Averaging 128-dim vectors)\n
6. **Real-time Inference** (Euclidean distance matching)\n
\n
### Pipeline Architecture:\n
```\n
TRAINING PHASE               INFERENCE PHASE\n
━━━━━━━━━━━━━━              ━━━━━━━━━━━━━━━\n
Video Capture (60s)          Webcam Frame\n
     ↓                            ↓\n
Face Detection (600+ frames)     Face Detection\n
     ↓                            ↓\n
FaceNet CNN Embeddings           FaceNet Embedding\n
     ↓                            ↓\n
Quality Scoring                  Distance Comparison\n
     ↓                            ↓\n
Embedding Averaging             Decision Threshold\n
     ↓                            ↓\n
Save Model                 AUTHORIZED/UNAUTHORIZED\n
```

## 1️⃣ Setup and Imports

In [ ]:
# Core AI/ML Libraries\n
import numpy as np\n
import pandas as pd\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
\n
# Computer Vision\n
import cv2\n
import face_recognition  # Built on dlib's FaceNet\n
from PIL import Image\n
\n
# Utilities\n
import json\n
import time\n
from pathlib import Path\n
from datetime import datetime, timedelta\n
from typing import List, Tuple\n
\n
# Configure visualization\n
plt.style.use('seaborn-v0_8-darkgrid')\n
plt.rcParams['figure.figsize'] = (14, 8)\n
plt.rcParams['font.size'] = 11\n
sns.set_palette('husl')\n
\n
print('✅ All libraries imported successfully!')\n
print(f'📅 Execution: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')\n
print(f'🧠 NumPy: {np.__version__}')\n
print(f'📷 OpenCV: {cv2.__version__}')\n
print(f'👤 face_recognition: Available')

## 2️⃣ AI Training Module: Video-Based Face Learning\n
\n
### Theory: Transfer Learning + Fine-Tuning\n
\n
**Pre-trained Model (FaceNet):**\n
- Trained on VGGFace2: 3.31 million images, 9,131 identities\n
- Architecture: Inception ResNet v1 (CNN)\n
- Output: 128-dimensional embedding vector\n
- Loss Function: Triplet loss (anchor-positive-negative)\n
\n
**Our Fine-Tuning Approach:**\n
- Capture 60-second video (600-1000 frames)\n
- User moves head → temporal variations\n
- Generate 128-dim embedding per frame\n
- **Average top-quality embeddings** = personalized model\n
\n
**Why Averaging Works:**\n
- Reduces noise from poor quality frames\n
- Creates robust representation across angles\n
- Maintains discriminative power\n
- No retraining needed (transfer learning!)\n
\n
$$\n
\\text{Final Embedding} = \\frac{1}{N} \\sum_{i=1}^{N} \\mathbf{e}_i\n
$$\n
\n
Where $\\mathbf{e}_i$ are top-N quality embeddings

In [ ]:
class AIFaceTrainer:\n
    """\n
    Dynamic AI Training Module\n
    \n
    Simulates 1-minute video capture with multi-angle face learning.\n
    In production: uses cv2.VideoCapture(0) for real webcam.\n
    """\n
    \n
    def __init__(self, user_name: str, duration_seconds: int = 60):\n
        self.user_name = user_name\n
        self.duration = duration_seconds\n
        self.encodings = []\n
        self.quality_scores = []\n
        self.timestamps = []\n
        \n
        print(f'🎓 AI Trainer initialized for: {user_name}')\n
        print(f'⏱️  Training duration: {duration_seconds}s')\n
    \n
    def capture_training_video(self, fps: int = 10) -> dict:\n
        """\n
        Simulate video capture with face detection + encoding generation.\n
        \n
        In production:\n
        - Opens webcam: cv2.VideoCapture(0)\n
        - Detects faces: face_recognition.face_locations()\n
        - Generates encodings: face_recognition.face_encodings()\n
        - Calculates quality: face_area / frame_area\n
        """\n
        num_frames = self.duration * fps\n
        \n
        print(f'\\n📹 Starting {self.duration}s video capture...')\n
        print(f'   Target: {num_frames} frames @ {fps} FPS\\n')\n
        \n
        for i in range(num_frames):\n
            # Progress updates\n
            if i % 60 == 0:\n
                progress = (i / num_frames) * 100\n
                elapsed = (i / num_frames) * self.duration\n
                print(f'   [{progress:5.1f}%] {elapsed:4.1f}s | Frame: {i}/{num_frames}')\n
            \n
            # Simulate FaceNet embedding (128-dim vector)\n
            # Real: face_recognition.face_encodings(frame)\n
            encoding = np.random.randn(128).astype(np.float32)\n
            \n
            # Simulate quality (0-1, based on face size, lighting, blur)\n
            quality = np.random.beta(a=5, b=2)  # Skewed towards higher quality\n
            \n
            # Simulate timestamp\n
            timestamp = (i / num_frames) * self.duration\n
            \n
            self.encodings.append(encoding)\n
            self.quality_scores.append(quality)\n
            self.timestamps.append(timestamp)\n
        \n
        print(f'\\n✅ Video capture complete!')\n
        print(f'   Frames collected: {len(self.encodings)}')\n
        \n
        return {\n
            'total_frames': len(self.encodings),\n
            'duration': self.duration,\n
            'avg_quality': float(np.mean(self.quality_scores)),\n
            'min_quality': float(np.min(self.quality_scores)),\n
            'max_quality': float(np.max(self.quality_scores))\n
        }\n
    \n
    def fine_tune_model(self, top_n: int = 50) -> np.ndarray:\n
        """\n
        Fine-tune model by averaging top-quality embeddings.\n
        \n
        This is the 'training' step:\n
        1. Sort frames by quality score\n
        2. Select top N frames (clear, well-lit, centered)\n
        3. Average their 128-dim embeddings\n
        4. Result = robust personalized face model\n
        """\n
        print(f'\\n🧠 Fine-tuning AI model...')\n
        print(f'   Method: Embedding Averaging')\n
        print(f'   Top frames: {top_n}\\n')\n
        \n
        # Sort by quality (descending)\n
        sorted_idx = np.argsort(self.quality_scores)[::-1]\n
        top_idx = sorted_idx[:top_n]\n
        \n
        # Get best embeddings\n
        top_encodings = np.array([self.encodings[i] for i in top_idx])\n
        top_qualities = np.array([self.quality_scores[i] for i in top_idx])\n
        \n
        print(f'   Selected: {len(top_encodings)} high-quality frames')\n
        print(f'   Quality range: [{top_qualities.min():.3f}, {top_qualities.max():.3f}]')\n
        print(f'   Mean quality: {top_qualities.mean():.3f}\\n')\n
        \n
        # Average embeddings (fine-tuning!)\n
        final_embedding = np.mean(top_encodings, axis=0)\n
        \n
        print(f'✅ Model fine-tuned successfully!')\n
        print(f'   Shape: {final_embedding.shape}')\n
        print(f'   Stats: μ={final_embedding.mean():.3f}, σ={final_embedding.std():.3f}')\n
        \n
        return final_embedding\n
\n
print('✅ AIFaceTrainer class defined')